# VGGT + Any4D Dynamic Mask Refinement on Colab

This notebook runs the inference-time experiment: VGGT initial geometry + Any4D scene flow + refined dynamic mask.

## 0. GPU check

In [ ]:
!nvidia-smi


## 1. Clone your repo

In [ ]:
%cd /content
!git clone https://github.com/ZexuZang/vggt-demo.git
%cd /content/vggt-demo
!git status


## 2. Install VGGT dependencies

In [ ]:
%cd /content/vggt-demo
!pip install -r requirements.txt
!pip install opencv-python matplotlib imageio


## 3. Clone and install Any4D

In [ ]:
%cd /content/vggt-demo
!mkdir -p third_party
![ -d third_party/Any4D ] || git clone https://github.com/Any-4D/Any4D.git third_party/Any4D
%cd /content/vggt-demo/third_party/Any4D
!pip install -e ".[all]"
!mkdir -p checkpoints
![ -f checkpoints/any4d_4v_combined.pth ] || wget -P checkpoints https://huggingface.co/airlabshare/any4d-checkpoint/resolve/main/any4d_4v_combined.pth


## 4. Prepare input frames

Put frames in `/content/vggt-demo/data/stroller_frames` using Colab upload or Google Drive.

In [ ]:
%cd /content/vggt-demo
!mkdir -p data/stroller_frames outputs
!find data/stroller_frames -maxdepth 1 -type f | head


## 5. Export VGGT predictions

In [ ]:
%cd /content/vggt-demo
!python scripts/any4d_mask/export_vggt_predictions.py \
  --image_dir data/stroller_frames \
  --output_npz outputs/vggt_pred.npz \
  --max_frames 8


## 6. Export Any4D predictions

In [ ]:
%cd /content/vggt-demo
!python scripts/any4d_mask/export_any4d_predictions.py \
  --any4d_root third_party/Any4D \
  --image_dir data/stroller_frames \
  --checkpoint_path third_party/Any4D/checkpoints/any4d_4v_combined.pth \
  --output_npz outputs/any4d_pred.npz \
  --ref_img_idx 0 \
  --start_idx 0 \
  --end_idx 8


## 7. Fuse Any4D motion cue with VGGT confidence

In [ ]:
%cd /content/vggt-demo
!python scripts/any4d_mask/refine_dynamic_mask.py \
  --vggt_npz outputs/vggt_pred.npz \
  --any4d_npz outputs/any4d_pred.npz \
  --image_dir data/stroller_frames \
  --output_dir outputs/refined_mask \
  --ref_idx 0 \
  --aggregate max \
  --dynamic_thresh 0.55 \
  --static_weight_thresh 0.45


## 8. Visualize outputs

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

out = Path('/content/vggt-demo/outputs/refined_mask')
for name in ['motion_probability.png','vggt_confidence.png','dynamic_probability.png','static_weight.png','refined_dynamic_mask.png','static_ba_mask.png','refined_dynamic_overlay.png','static_ba_overlay.png']:
    p = out / name
    if p.exists():
        print(name)
        display(Image.open(p))


## 9. Optional: push branch from Colab

Only run this after configuring GitHub authentication. Do not paste tokens into notebook outputs.

In [ ]:
# !git config user.name "ZexuZang"
# !git config user.email "zexuzang1@gmail.com"
# !git checkout -b any4d-dynamic-mask
# !git add scripts/any4d_mask notebooks/VGGT_Any4D_Dynamic_Mask_Colab.ipynb docs/VGGT_ANY4D_DYNAMIC_MASK.md
# !git commit -m "Add Any4D dynamic mask refinement for VGGT"
# !git push origin any4d-dynamic-mask
